# 06 — Classification Binaire vs Multi-classe (LIAR Dataset)

## Objectif

Comparer les performances des modèles selon le nombre de classes :

| Configuration | Classes | Approche |
|---|---|---|
| **3 classes** | fake / nuanced / real | Baseline existant |
| **2 classes** | fake / real | Filtrage des exemples "nuanced" |

## Hypothèses

1. **Les modèles binaires devraient mieux performer** — la classe "nuanced" est intrinsèquement ambiguë (mi-vraie mi-fausse) et crée une frontière floue avec les deux autres classes.
2. **L'avantage binaire sera surtout visible sur le F1 fake** — en 3 classes, les modèles confondent souvent fake et nuanced.

> La configuration retenue ici sera utilisée dans le notebook **07** pour l'évaluation out-of-domain sur WELFake.

## 0. Imports

In [20]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from preprocessing import load_liar, preprocess_liar
from features import TfidfFeatures, CombinedFeatures, META_COLS
from train import train_logistic_regression, train_random_forest, train_xgboost
from welfake_loader import load_welfake, preprocess_welfake, add_dummy_metadata

print("Imports OK.")

Imports OK.


## 1. Chargement et préparation des données

### 1.1 Version 3 classes (baseline)

Encodage original : **fake=0, nuanced=1, real=2**

In [21]:
# --- Chargement LIAR ---
train_raw, valid_raw, test_raw = load_liar('data/raw')
train_3 = preprocess_liar(train_raw)
test_3  = preprocess_liar(test_raw)

y_train_3 = train_3['label_encoded'].values   # {0, 1, 2}
y_test_3  = test_3['label_encoded'].values

print("=== Version 3 classes ===")
print(f"Train : {len(train_3)} exemples")
print(f"Test  : {len(test_3)} exemples")
print(f"\nDistribution train :")
for cls, name in [(0,'fake'),(1,'nuanced'),(2,'real')]:
    n = (y_train_3 == cls).sum()
    print(f"  {name:<10}: {n:>5}  ({n/len(y_train_3)*100:.1f}%)")

LIAR chargé — train: 10240, valid: 1284, test: 1267
=== Version 3 classes ===
Train : 10240 exemples
Test  : 1267 exemples

Distribution train :
  fake      :  2834  (27.7%)
  nuanced   :  3768  (36.8%)
  real      :  3638  (35.5%)


### 1.2 Version 2 classes (binaire)

On **filtre** les exemples "nuanced" (label=1) et on **remémorise** :  
fake = 0 → **0** | real = 2 → **1**

In [22]:
def make_binary(df):
    """Filtre les nuanced et remémorise fake=0 / real=1."""
    df_bin = df[df['label_encoded'] != 1].copy()
    # 0 reste 0 (fake), 2 devient 1 (real)
    df_bin['label_binary'] = df_bin['label_encoded'].map({0: 0, 2: 1})
    return df_bin

train_2 = make_binary(train_3)
test_2  = make_binary(test_3)

y_train_2 = train_2['label_binary'].values   # {0, 1}
y_test_2  = test_2['label_binary'].values

print("=== Version 2 classes (binaire) ===")
print(f"Train : {len(train_2)} exemples  (vs {len(train_3)} en 3 classes, -{len(train_3)-len(train_2)} nuanced supprimés)")
print(f"Test  : {len(test_2)} exemples   (vs {len(test_3)})")
print(f"\nDistribution train :")
for cls, name in [(0,'fake'),(1,'real')]:
    n = (y_train_2 == cls).sum()
    print(f"  {name:<10}: {n:>5}  ({n/len(y_train_2)*100:.1f}%)")

=== Version 2 classes (binaire) ===
Train : 6472 exemples  (vs 10240 en 3 classes, -3768 nuanced supprimés)
Test  : 790 exemples   (vs 1267)

Distribution train :
  fake      :  2834  (43.8%)
  real      :  3638  (56.2%)


## 2. Extraction des features

On utilise le même pipeline TF-IDF + métadonnées pour les deux versions.  
**Important** : deux TF-IDF distincts sont entraînés, chacun sur son propre corpus d'entraînement.

In [23]:
# ---- Features version 3 classes ----
tfidf_3    = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf_3.fit_transform(train_3['statement_clean'])
combined_3 = CombinedFeatures(text_features=tfidf_3, meta_cols=META_COLS)
X_train_3  = combined_3.fit_transform(train_3)
X_test_3   = combined_3.transform(test_3)

# ---- Features version 2 classes (TF-IDF entraîné sur corpus réduit) ----
tfidf_2    = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf_2.fit_transform(train_2['statement_clean'])
combined_2 = CombinedFeatures(text_features=tfidf_2, meta_cols=META_COLS)
X_train_2  = combined_2.fit_transform(train_2)
X_test_2   = combined_2.transform(test_2)

print(f"X_train_3 : {X_train_3.shape}  |  X_test_3 : {X_test_3.shape}")
print(f"X_train_2 : {X_train_2.shape}  |  X_test_2 : {X_test_2.shape}")

TF-IDF fit — vocabulaire : 9049 features, 10240 documents
TF-IDF fit — vocabulaire : 5912 features, 6472 documents
X_train_3 : (10240, 9053)  |  X_test_3 : (1267, 9053)
X_train_2 : (6472, 5916)  |  X_test_2 : (790, 5916)


## 3. Entraînement des modèles

In [24]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

def train_xgboost_binary(X_train, y_train):
    """XGBoost avec objectif binaire (fake=0 / real=1)."""
    print("Entraînement XGBoost (binaire)...")
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    sample_weight = np.array([weights[y] for y in y_train])
    X_dense = X_train.toarray() if hasattr(X_train, "toarray") else X_train
    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
        verbosity=0,
    )
    model.fit(X_dense, y_train, sample_weight=sample_weight)
    print("  XGBoost binaire — terminé")
    return model

print("=== Entraînement version 3 classes ===")
lr_3  = train_logistic_regression(X_train_3, y_train_3)
rf_3  = train_random_forest(X_train_3, y_train_3, n_estimators=300)
xgb_3 = train_xgboost(X_train_3, y_train_3)

print("\n=== Entraînement version 2 classes ===")
lr_2  = train_logistic_regression(X_train_2, y_train_2)
rf_2  = train_random_forest(X_train_2, y_train_2, n_estimators=300)
xgb_2 = train_xgboost_binary(X_train_2, y_train_2)   # objectif binaire

print("\nTous les modèles entraînés.")

=== Entraînement version 3 classes ===
Entraînement Logistic Regression...
  LR — terminé
Entraînement Random Forest...
  RF — terminé
Entraînement XGBoost...
  XGBoost — terminé

=== Entraînement version 2 classes ===
Entraînement Logistic Regression...
  LR — terminé
Entraînement Random Forest...
  RF — terminé
Entraînement XGBoost (binaire)...
  XGBoost binaire — terminé

Tous les modèles entraînés.


## 4. Évaluation in-domain

### 4.1 Fonctions d'évaluation adaptées

On définit deux fonctions distinctes : une pour 3 classes, une pour 2 classes.

In [25]:
def eval_3class(y_true, y_pred, model_name):
    """Évaluation multi-classe (fake=0, nuanced=1, real=2)."""
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    f1c = f1_score(y_true, y_pred, average=None, labels=[0,1,2], zero_division=0)
    print(f"\n{'='*50}")
    print(f"  [3 classes] {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}  |  F1 macro : {f1:.4f}")
    print(classification_report(y_true, y_pred,
          target_names=['fake','nuanced','real'], labels=[0,1,2], zero_division=0))
    return {
        'model': model_name, 'mode': '3 classes',
        'accuracy': round(acc,4), 'f1_macro': round(f1,4),
        'f1_fake': round(f1c[0],4), 'f1_nuanced': round(f1c[1],4), 'f1_real': round(f1c[2],4),
    }

def eval_2class(y_true, y_pred, model_name):
    """Évaluation binaire (fake=0, real=1)."""
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    f1c = f1_score(y_true, y_pred, average=None, labels=[0,1], zero_division=0)
    print(f"\n{'='*50}")
    print(f"  [2 classes] {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}  |  F1 macro : {f1:.4f}")
    print(classification_report(y_true, y_pred,
          target_names=['fake','real'], labels=[0,1], zero_division=0))
    return {
        'model': model_name, 'mode': '2 classes',
        'accuracy': round(acc,4), 'f1_macro': round(f1,4),
        'f1_fake': round(f1c[0],4), 'f1_nuanced': None, 'f1_real': round(f1c[1],4),
    }

print("Fonctions d'évaluation définies.")

Fonctions d'évaluation définies.


### 4.2 Résultats in-domain — 3 classes

In [26]:
results_in_3 = []
X_test_3_dense = X_test_3.toarray() if hasattr(X_test_3, 'toarray') else X_test_3

results_in_3.append(eval_3class(y_test_3, lr_3.predict(X_test_3), 'LR'))
results_in_3.append(eval_3class(y_test_3, rf_3.predict(X_test_3), 'RF'))
results_in_3.append(eval_3class(y_test_3, xgb_3.predict(X_test_3_dense), 'XGBoost'))


  [3 classes] LR
  Accuracy : 0.5375  |  F1 macro : 0.5442
              precision    recall  f1-score   support

        fake       0.61      0.69      0.65       341
     nuanced       0.51      0.43      0.47       477
        real       0.50      0.53      0.52       449

    accuracy                           0.54      1267
   macro avg       0.54      0.55      0.54      1267
weighted avg       0.53      0.54      0.53      1267


  [3 classes] RF
  Accuracy : 0.5651  |  F1 macro : 0.5697
              precision    recall  f1-score   support

        fake       0.62      0.67      0.64       341
     nuanced       0.54      0.45      0.49       477
        real       0.55      0.61      0.58       449

    accuracy                           0.57      1267
   macro avg       0.57      0.58      0.57      1267
weighted avg       0.56      0.57      0.56      1267


  [3 classes] XGBoost
  Accuracy : 0.5556  |  F1 macro : 0.5577
              precision    recall  f1-score   support

### 4.3 Résultats in-domain — 2 classes

In [27]:
results_in_2 = []
X_test_2_dense = X_test_2.toarray() if hasattr(X_test_2, 'toarray') else X_test_2

results_in_2.append(eval_2class(y_test_2, lr_2.predict(X_test_2), 'LR'))
results_in_2.append(eval_2class(y_test_2, rf_2.predict(X_test_2), 'RF'))
results_in_2.append(eval_2class(y_test_2, xgb_2.predict(X_test_2_dense), 'XGBoost'))


  [2 classes] LR
  Accuracy : 0.7899  |  F1 macro : 0.7867
              precision    recall  f1-score   support

        fake       0.75      0.77      0.76       341
        real       0.82      0.80      0.81       449

    accuracy                           0.79       790
   macro avg       0.79      0.79      0.79       790
weighted avg       0.79      0.79      0.79       790


  [2 classes] RF
  Accuracy : 0.7873  |  F1 macro : 0.7837
              precision    recall  f1-score   support

        fake       0.75      0.76      0.76       341
        real       0.82      0.81      0.81       449

    accuracy                           0.79       790
   macro avg       0.78      0.78      0.78       790
weighted avg       0.79      0.79      0.79       790


  [2 classes] XGBoost
  Accuracy : 0.7747  |  F1 macro : 0.7723
              precision    recall  f1-score   support

        fake       0.72      0.78      0.75       341
        real       0.82      0.77      0.80       44

### 4.4 Tableau comparatif in-domain

In [28]:
df_in_3 = pd.DataFrame(results_in_3)
df_in_2 = pd.DataFrame(results_in_2)

print(f"\n{'Modèle':<10} | {'F1 macro (3 cls)':>18} | {'F1 macro (2 cls)':>18} | {'Gain':>8}")
print('-' * 65)
for m in ['LR', 'RF', 'XGBoost']:
    f1_3 = df_in_3[df_in_3['model']==m]['f1_macro'].values[0]
    f1_2 = df_in_2[df_in_2['model']==m]['f1_macro'].values[0]
    gain = (f1_2 - f1_3) * 100
    print(f"  {m:<8} | {f1_3:>18.4f} | {f1_2:>18.4f} | {gain:>+7.2f} pp")


Modèle     |   F1 macro (3 cls) |   F1 macro (2 cls) |     Gain
-----------------------------------------------------------------
  LR       |             0.5442 |             0.7867 |  +24.25 pp
  RF       |             0.5697 |             0.7837 |  +21.40 pp
  XGBoost  |             0.5577 |             0.7723 |  +21.46 pp


## 5. Visualisations interactives (Plotly)

Deux graphiques pour comparer les performances in-domain :

1. **F1 macro** : comparaison globale par modèle (2 classes vs 3 classes)
2. **F1 par classe** : focus sur fake et real

### 5.1 F1 macro — 2 classes vs 3 classes

In [29]:
# --- Graphique 1 : F1 macro in-domain — 2 classes vs 3 classes ---

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    name="3 classes (fake / nuanced / real)",
    x=MODELS,
    y=[df_in_3[df_in_3['model']==m]['f1_macro'].values[0] for m in MODELS],
    marker_color="#5b7fa6",
    opacity=0.85,
    text=[f"{df_in_3[df_in_3['model']==m]['f1_macro'].values[0]:.3f}" for m in MODELS],
    textposition="outside",
))

fig1.add_trace(go.Bar(
    name="2 classes (fake / real)",
    x=MODELS,
    y=[df_in_2[df_in_2['model']==m]['f1_macro'].values[0] for m in MODELS],
    marker_color="#27ae60",
    opacity=0.85,
    text=[f"{df_in_2[df_in_2['model']==m]['f1_macro'].values[0]:.3f}" for m in MODELS],
    textposition="outside",
))

fig1.update_layout(
    title=dict(text="F1 macro In-domain — 3 classes vs 2 classes (LIAR test set)", font=dict(size=16)),
    barmode="group",
    yaxis=dict(title="F1 macro", range=[0, 1.0], tickformat=".2f"),
    xaxis_title="Modèle",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white",
    height=460,
)

fig1.show()

### 5.2 F1 par classe — fake et real

In [30]:
# --- Graphique 2 : F1 fake et F1 real — comparaison 2 vs 3 classes in-domain ---

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["F1 — classe fake", "F1 — classe real"],
    shared_yaxes=True,
)

colors_3cls = "#5b7fa6"
colors_2cls = "#27ae60"

for col_idx, (metric_3, metric_2, title) in enumerate([
    ('f1_fake', 'f1_fake', 'fake'),
    ('f1_real', 'f1_real', 'real'),
], start=1):
    vals_3 = [df_in_3[df_in_3['model']==m][metric_3].values[0] for m in MODELS]
    vals_2 = [df_in_2[df_in_2['model']==m][metric_2].values[0] for m in MODELS]

    fig2.add_trace(go.Bar(
        name="3 classes" if col_idx == 1 else None,
        showlegend=(col_idx == 1),
        x=MODELS, y=vals_3,
        marker_color=colors_3cls, opacity=0.85,
        text=[f"{v:.3f}" for v in vals_3], textposition="outside",
        legendgroup="3cls",
    ), row=1, col=col_idx)

    fig2.add_trace(go.Bar(
        name="2 classes" if col_idx == 1 else None,
        showlegend=(col_idx == 1),
        x=MODELS, y=vals_2,
        marker_color=colors_2cls, opacity=0.85,
        text=[f"{v:.3f}" for v in vals_2], textposition="outside",
        legendgroup="2cls",
    ), row=1, col=col_idx)

fig2.update_layout(
    title=dict(text="F1 par classe in-domain — 2 classes vs 3 classes", font=dict(size=16)),
    barmode="group",
    yaxis=dict(title="F1", range=[0, 1.0], tickformat=".2f"),
    template="plotly_white",
    height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)

fig2.show()

## 6. Sauvegarde des résultats in-domain

In [31]:
rows = []
for r in results_in_3:
    rows.append(r | {'domain': 'in-domain (LIAR)', 'mode': '3 classes'})
for r in results_in_2:
    rows.append(r | {'domain': 'in-domain (LIAR)', 'mode': '2 classes'})

df_save = pd.DataFrame(rows)
os.makedirs('outputs', exist_ok=True)
df_save.to_csv('outputs/results_binary_vs_multiclass.csv', index=False)
print('Résultats sauvegardés → outputs/results_binary_vs_multiclass.csv')
df_save

Résultats sauvegardés → outputs/results_binary_vs_multiclass.csv


,model,mode,accuracy,f1_macro,f1_fake,f1_nuanced,f1_real,domain
0,LR,3 classes,0.5375,0.5442,0.6465,0.4705,0.5156,in-domain (LIAR)
1,RF,3 classes,0.5651,0.5697,0.6431,0.4874,0.5786,in-domain (LIAR)
2,XGBoost,3 classes,0.5556,0.5577,0.6239,0.4824,0.5668,in-domain (LIAR)
3,LR,2 classes,0.7899,0.7867,0.7608,NaN,0.8126,in-domain (LIAR)
4,RF,2 classes,0.7873,0.7837,0.7558,NaN,0.8117,in-domain (LIAR)
5,XGBoost,2 classes,0.7747,0.7723,0.7493,NaN,0.7954,in-domain (LIAR)
